# 1. The run

A run is a conversation with a loop around it. Each round the runtime appends one system
notification (what settled, what failed, what was refused), renders the record into
messages, and calls the model once. The model answers in prose with LNDL in it: `<lvar>` names a
value, `<lact>` asks a command to run, `OUT{}` finishes. Handlers do the work; the record remembers
everything; the model never sees a value twice, it points at it.

This notebook uses a scripted backend, so the model is four strings written below and every turn is
readable. Nothing here needs a key. The design is recorded in `docs/adr` (ADR-0001 to ADR-0007).
It is stored with its outputs cleared; run it to see what each cell prints.

In [ ]:
import tempfile
from pathlib import Path

from pydantic import BaseModel, Field

from lionagi import Actor, Context, Kind, Profile, RunRequest, Success, lndl

root = Path(tempfile.mkdtemp())
(root / "poem.txt").write_text(
    "the loop goes round\nand every round is one message in\nand one message out\n"
)
(root / "notes.txt").write_text("handlers act, the record remembers\n")
(root / "todo.md").write_text("- write the notebooks\n")
sorted(p.name for p in root.iterdir())

## Specs are what the model may say

A Spec is a pydantic model. Its docstring and field descriptions are the whole of what the model
is told about the command, so they are written for the model. A handler is a function registered
on an actor for one Spec; registering is what puts the Spec in the actor's Operable.

In [ ]:
class ListDir(BaseModel):
    """List the files in the working directory."""


class ReadFile(BaseModel):
    """Read one file from the working directory. Returns its text."""

    path: str = Field(description="file name")


class WordCount(BaseModel):
    """Count the words in a text. Point at a value you already have: text=*r1"""

    text: str


class Report(BaseModel):
    """What you found."""

    summary: str = Field(description="one sentence")
    longest: str = Field(description="the file with the most words")
    words: int = Field(description="its word count")


actor = Actor("librarian", privileges={"fs.read"})


@actor.handler(ListDir, requires={"fs.read"})
async def list_dir(req: ListDir, ctx: Context) -> list[str]:
    return sorted(p.name for p in root.iterdir())


@actor.handler(ReadFile, requires={"fs.read"})
async def read_file(req: ReadFile, ctx: Context) -> str:
    return (root / req.path).read_text()


@actor.handler(WordCount)
async def word_count(req: WordCount, ctx: Context) -> int:
    return len(req.text.split())


print(actor.operable.guidance())

Seven commands came with the actor: `context.hide`, `context.show`, `context.summarize`,
`note.list`, `note.find`, `note.get`, `note.delete`. They are the model's own hands on its view
and its notes.

The LNDL guidance is generated too. This is the part of the system prompt that teaches the
three constructs, with this run's output named in the example.

In [ ]:
print(lndl.guidance(("report",)))

## A profile is a hat

A profile names which of the actor's commands the model is told about, the privileges the run
acts under (a subset of the actor's), and the role's prose. It has no address and no credentials;
the actor has those. The prompt is generated from the profile's subset, so a command outside it is
never presented: that is the first gate, decided when the run starts.

In [ ]:
librarian = Profile(
    name="librarian",
    privileges=frozenset({"fs.read"}),
    system="You answer questions about the files in the working directory. Work in rounds; point at values.",
)

## Four turns, written in advance

Read them against the record below. Round 1 lists. Round 2 reads two files in one turn: the calls
run concurrently and both results arrive in the next notification. Round 3 names a note (`note.`
aliases persist across runs), counts words by pointing at the texts, and hides the two file bodies
from the view. Round 4 declares the report's fields by name (`<lvar report.summary s1>`), asks two
more counts, and finishes. Every `<lact>` in a turn runs, `OUT{}` or not; `OUT{}` waits only for
what it names or points at, then the fields assemble, and once they are accepted whatever is still
running is cancelled.

In [ ]:
def show(run, width=140):
    """The record, one entry per line: what came in, what the system and the model said, what settled."""
    print(f"run {run.id}: {type(run.outcome).__name__} after {run.round} rounds\n")
    for e in run.record:
        if e.kind is Kind.RESULT:
            res = e.content
            body = f"{e.alias} <- {res.spec}: " + (res.preview(width) if res.ok else "ERR " + str(res.error))
        else:
            body = str(e.content)
        print(f"[{e.seq:>2} {e.name:>5} r{e.round}] {body}\n")


def scripted(*turns):
    """A backend whose turns are written in advance: the view comes in, the next string goes out."""
    it = iter(turns)

    async def backend(messages):
        return next(it)

    return backend

In [ ]:
turns = [
    "I will start by listing the directory.\n<lact l1>list_dir()</lact>",
    '<lact r1>read_file(path="poem.txt")</lact>\n<lact r2>read_file(path="notes.txt")</lact>',
    "<lvar note.seen>poem.txt and notes.txt read; the poem looks longer</lvar>\n"
    "<lact w1>word_count(text=*r1)</lact>\n<lact w2>word_count(text=*r2)</lact>\n"
    '<lact h1>context.hide(refs=["r1", "r2"])</lact>',
    "<lvar report.summary s1>Three files; the poem is the longest.</lvar>\n"
    "<lvar report.longest lg1>poem.txt</lvar>\n"
    "<lact report.words w3>word_count(text=*r1)</lact>\n"
    "<lact w4>word_count(text=*r2)</lact>\n"
    "OUT{report: [s1, lg1, w3]}",
]

request = RunRequest(inputs=("Which file has the most words?",), emits=(Report,), max_rounds=6)
run = await actor.run(librarian, request, scripted(*turns))
show(run)

Things to notice in the record:

- Every entry has a name (`_in0`, `_sys1`, `_m2`, `_r4`) and the model's aliases (`l1`, `r1`) resolve
  to entries too, so a pointer can name either.
- The notification says what settled with a type, a size and a preview. The value itself is never
  repeated; the model points at it.
- Round 3's `context.hide` took `r1` and `r2` out of the view. They stay on the record, and
  `*r1` still resolves in round 4.
- The last turn carried `OUT{}` beside two commands. Both ran (`w3`, `w4`); `OUT{}` waited for
  `w3`, which it names, and the run ended `Success`. `w4` finished in time; one still running then
  would have been cancelled, so a model that writes "edit, then done" in one turn points `OUT{}`
  at the edit (ADR-0002).
- The record ends with the closure, the SYSTEM entry that carries the outcome and names any
  command the end cancelled.

In [ ]:
assert isinstance(run.outcome, Success)
report = run.output["report"]
print(type(report).__name__, report.model_dump())
print("w4 ran too:", run.resolve("w4"), "| note.seen resolves to:", run.resolve("note.seen"))
for n in actor.notes_for(librarian).list():
    print("note", n.key, "=", n.value, f"(run {n.run}, v{n.version})")

A backend is any callable from a list of messages to text, `Backend = Callable[[list[Message]],
Awaitable[str] | str]`. The scripted one above ignores its input. The real ones live in
`hub.backend`: OpenRouter, Codex, and Claude Code as one call per round or one resumed session.
`examples/backends.py` runs the same loop on each of them.